## Feature Preprocessing and Correlations
We prepare the feature matrix for analyses, specifying various article-level features (each row is an article, features are columns).<br>
For continuous features, we check for normality and skewness, and apply a transformation only where it improves the distribution. The three continuous features are treated differently, for different reasons:
- **Publication age** (weeks since publication) is $log$-transformed on *functional* rather than distributional grounds: citation accumulation is multiplicative, so a $log$-scale linearises the age-citation relationship. This follows the recommendations of `Thelwall and Wilson (2014)` and `Clauset et al. (2009)`.
- **Number of authors** is $log$-transformed on *distributional* grounds: it is strongly right-skewed in its natural scale, and the $log$-transform renders it close to symmetric.
- **Venue impact** (2-year mean citedness) is kept in its **natural scale**: it is only mildly skewed, and a $log$-transform over-corrects it into a pronounced *left* skew. See the skewness check below.
<br><br>
We then compute pairwise correlations between features and the variance inflation factors (VIF) for each feature. Features with high VIF values (e.g., > 5 or 10) may indicate multicollinearity and may need to be removed or combined in subsequent analyses.
<br><br>
The final feature matrix includes the following features (columns) for each article (row):
- Is Sharing Data (binary)
- Sharing Class (ordinal)
- Has US Author (binary)
- Is Open Access (binary)
- Has Preprint (binary)
- Venue Impact (continuous)
- log(Weeks Since Pub.) (continuous)
- log(Number of Authors) (continuous)


_References:_
- Clauset, A., Shalizi, C. R., & Newman, M. E. (2009). Power-law distributions in empirical data. SIAM review, 51(4), 661-703.
- Thelwall, M., & Wilson, P. (2014). Regression for citation data: An evaluation of different methods. Journal of Informetrics, 8(4), 963-971.

In [ ]:
from typing import Literal, Dict
from itertools import combinations
from copy import deepcopy

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers import dataset
from helpers.config import (
    CLASS_COLORS, SHARING_CLASS_ORDER, BINARY_FEATURES,
    TITLE_FONT, AXIS_TITLE_FONT, AXIS_TICK_FONT, LEGEND_FONT, FONT_FAMILY,
    VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME,
)
from helpers.plotting import save_figure
from helpers.stats import compare_binary, compare_continuous

pio.renderers.default = "browser"

combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

### Skewness
Check the skewness of continuous features with and without $log$-transformation. A skewness value greater than 1 or less than -1 indicates a highly skewed distribution, while values between -1 and 1 indicate a moderately skewed distribution.

In [ ]:
print(f"NumAuthors skewness:\tNatural Scale: {stats.skew(combined["NumAuthors"]) :.4f}\tLog Scale: {stats.skew(np.log(combined["NumAuthors"])) :.4f}\tLog+1 Scale: {stats.skew(np.log(combined["NumAuthors"] + 1)) :.4f}")
print(f"{VENUE_IMPACT_METRIC} skewness:\tNatural Scale {stats.skew(combined[VENUE_IMPACT_METRIC]) :.4f}\tLog Scale: {stats.skew(np.log(combined[VENUE_IMPACT_METRIC])) :.4f}\tLog+1 Scale: {stats.skew(np.log(combined[VENUE_IMPACT_METRIC] + 1)) :.4f}")

### Build Feature Matrix

In [ ]:
# assembled by `helpers.dataset.build_feature_matrix()`
FEATURES_DF.head()

### Pairwise Correlations
We calculate the pairwise correlations between article features, along with matching statistical test for significance, based on the types of the two features being compared:
- Binary-Binary: Pearson's `phi` coefficient (equivalent to Pearson r on 0/1 data) with Chi-square or Fisher's exact test for significance.
- Binary-Continuous: Point-biserial correlation with t-test (with $dof=N-1$) for significance.
- Continuous-Continuous: Pearson correlation with t-test for significance.

In this analysis, we exclude the `Sharing Class` feature, which is an ordinal refinement of `Is Sharing Data` and is therefore collinear by construction. We only include `Is Sharing Data` in the correlation matrix and VIF calculations.
<br><br>
To Visualize the results, we generate a heatmap of the pairwise correlation coefficients, with the upper-triangle of the matrix filled and thediagonal and lower-triangle left blank (NaN). We do not include the p-values in the heatmap, as this is a diagnostic for collinearity among the regression predictors, and only the magnitude of each coefficient matters (still, the uncorrected p-values are retained in the `pairs` dataframe for inspection only).

In [ ]:
def pairwise_correlation(x, y, x_is_binary, y_is_binary):
    """Coefficient + p-value using the measure/test matched to the pair's variable types."""
    paired = pd.concat([x, y], axis=1).dropna()
    a, b = paired.iloc[:, 0], paired.iloc[:, 1]
    if x_is_binary and y_is_binary:                      # binary-binary -> phi + chi-square / Fisher
        table = pd.crosstab(a, b)
        _, p, _, expected = stats.chi2_contingency(table, correction=False)
        if (expected < 5).any():
            _, p = stats.fisher_exact(table)
        coefficient = np.corrcoef(a, b)[0, 1]            # phi == Pearson r on 0/1 data
        method = "phi (chi2/Fisher)"
    elif x_is_binary or y_is_binary:                     # binary-continuous -> point-biserial
        binary, continuous = (a, b) if x_is_binary else (b, a)
        coefficient, p = stats.pointbiserialr(binary, continuous)
        method = "point-biserial"
    else:                                                # continuous-continuous -> Pearson
        coefficient, p = stats.pearsonr(a, b)
        method = "pearson"
    return coefficient, p, method


def significance_class(q):
    if pd.isnull(q) or q < 0:
        return np.nan
    if q < 0.001:
        return "***"
    if q < 0.01:
        return "**"
    if q < 0.05:
        return "*"
    return "n.s."

In [ ]:
CORRELATION_FEATURES = [col for col in FEATURES_DF.columns if col != "Sharing Class"]
correlation_df = FEATURES_DF[CORRELATION_FEATURES]
feature_names = list(correlation_df.columns)
n_features = len(feature_names)
coef_matrix = np.full((n_features, n_features), np.nan)
np.fill_diagonal(coef_matrix, 1.0)
pair_rows = []
for i in range(n_features):
    for j in range(i + 1, n_features):
        xi, xj = feature_names[i], feature_names[j]
        r, p, method = pairwise_correlation(
            correlation_df[xi], correlation_df[xj], xi in BINARY_FEATURES, xj in BINARY_FEATURES
        )
        coef_matrix[i, j] = coef_matrix[j, i] = r
        pair_rows.append({"i": i, "j": j, "X": xi, "Y": xj, "method": method, "r": r, "p_unc": p})

# NOTE: we calculate significance for these correlations but don't render in figures or report in
# the main article. This matrix is a *diagnostic* for collinearity among the regression predictors,
# so only the magnitude of each coefficient matters (see also the VIF table below).
pairs = pd.DataFrame(pair_rows)
pairs["p_fdr"] = multipletests(pairs["p_unc"], method="fdr_bh")[1]
pairs["sig"] = pairs["p_fdr"].map(significance_class)
pairs[["X", "Y", "method", "r", "p_unc", "p_fdr", "sig"]].round(4)

In [ ]:
# N x N heatmap showing the diagonal and the upper triangle; the lower triangle is the mirror
# image and carries no extra information, so it is left blank (NaN -> uncolored). Keeping the
# diagonal (self-correlations, r = 1) means every row and column has at least one populated cell.
cell_text = [["" for _ in range(n_features)] for _ in range(n_features)]
z_color = coef_matrix.copy()
z_color[np.tril_indices(n_features, k=-1)] = np.nan     # blank the STRICT lower triangle
for i in range(n_features):
    for j in range(i, n_features):                      # start at i to include the diagonal
        cell_text[i][j] = f"{coef_matrix[i, j]:.2f}"

corr_fig = go.Figure(go.Heatmap(
    z=z_color,
    x=[feat.replace("log", "<i>log</i>") for feat in feature_names],
    y=[feat.replace("log", "<i>log</i>") for feat in feature_names],
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1, xgap=2, ygap=2,
    text=cell_text, texttemplate="%{text}", textfont=AXIS_TICK_FONT,
    colorbar=dict(
        title=dict(text="<i>r</i>", font=AXIS_TITLE_FONT),
        tickfont=AXIS_TICK_FONT, thickness=14, len=0.85,
    ),
    hovertemplate="%{y} \u00d7 %{x}<extra></extra>",
))

# update layout and styling
corr_fig.update_yaxes(autorange="reversed", tickfont=AXIS_TICK_FONT)
corr_fig.update_xaxes(side="top", tickangle=-35, tickfont=AXIS_TICK_FONT)
# `mirror=True` repeats each axis line on the opposite side, closing the box on all four edges
corr_fig.update_xaxes(showline=True, linewidth=1, linecolor="black", mirror=True)
corr_fig.update_yaxes(showline=True, linewidth=1, linecolor="black", mirror=True)
corr_fig.update_layout(
    width=720, height=600,
    # `yref="container"` measures y against the whole figure rather than the plotting area, so the
    # title sits at a predictable offset from the top edge; `margin.t` then reserves room for both
    # the title and the angled x tick labels drawn beneath it.
    title=dict(
        text="<b>Pairwise Feature Correlations</b>", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.97, yanchor="top", yref="container",
    ),
    margin=dict(t=160, b=10, l=10, r=10),
    template="simple_white",
)

if False:
    corr_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(corr_fig, "corr_fig.png", width=720, height=600)

### Variance Inflation Factor
The variance inflation factor ([VIF](https://en.wikipedia.org/wiki/Variance_inflation_factor)) measures multi-collinearity between regression predictors.<br>
If predictor $X_i$ has a $VIF(X_i) = 1$ - it indicates that the predictor is completely independent of **all** other predictors in the model. Conversely, high values, e.g. $VIF(X_i) > 5$ indicate large multi-collinearity in the model, suggesting that some predictors should be excluded or pre-processed\* before adding them to the model.<br>
(\*pre processing predictors = recentering, normalizing, or combining two or more variables to a single one)

In [ ]:
vif_input = sm.add_constant(correlation_df)
vif = pd.DataFrame({
    "variable": vif_input.columns,
    "VIF": [variance_inflation_factor(vif_input.values, i).round(4) for i in range(vif_input.shape[1])]
})
display(vif)

## Relationship between Data-Sharing and Other Features
We run a set of statistical tests to examine if data-sharing is associated with other features of the publications in our dataset, such as publication age, number of authors, and impact score.
<br><br>
**Continues Features**<br>
Comparison between _sharing_ and _non-sharing_ articles is done using a non-parametric Mann-Whitney U-test or, if permitted, a parametric two-sided independent t-test. Comparison between sharing classes (_fixation_-, _trial_- and _participant_-level data) is done using a non-parametric Kruskal-Wallis test or, if permitted, a parametric one-way ANOVA.<br>
To chose between the two, we check for normality of the distributions using a Shapiro-Wilk test. If both distributions are normal, we use a t-test; otherwise, we use a Mann-Whitney U-test.
<br><br>
**Binary Features**<br>
We use a Chi-squared test of independence (or Fisher's exact test for small samples) to examine the relationship between data-sharing and binary features, such as whether the article has a US author, is open-access, or has a pre-print.
<br><br>
**Multiple Comparisons**<br>
We control for multiple comparisons across all tests using the Binyamini-Hochberg procedure, which controls the false discovery rate (FDR) at 0.05.

The `compare_binary()` and `compare_continuous()` helpers used below live in `helpers/stats.py`, since notebook 03 uses them too.

In [ ]:
_SUBTITLE_MAPPING = {
    "Has US Author": "Has U.S. Author",
    "Venue Impact": VENUE_IMAPCT_METRIC_NAME,
    "log(Weeks Since Pub.)": "Publication Age",
    "log(Number of Authors)": "Number of Authors",
}
_Y_AXIS_MAPPING = {
    "Venue Impact": "Impact Score",
    "log(Weeks Since Pub.)": "<i>log</i>(weeks)",
    "log(Number of Authors)": "<i>log</i>(count)"
}
_EFFECT_NOTATION = {
    "Cramer V": lambda v: f"Cramer's $V = {v:.2f}$",
    "partial eta2": lambda v: f"$\\eta_{{p}}^{{2}} = {v:.2f}$",
    "epsilon2": lambda v: f"$\\epsilon^{{2}} = {v:.2f}$",
    "rank-biserial": lambda v: f"$r_{{rb}} = {v:.2f}$",
    "Cohen d": lambda v: f"Cohen's $d = {v:.2f}$",
    "CLES": lambda v: f"$CLES = {v:.2f}$",
    "Odds Ratio": lambda v: f"$OR = {v:.2f}$",
}


def plot_feature_comparison(data, group_feature, group_order, results_df, main_title, palette=CLASS_COLORS):
    """2x3 grid comparing groups across the 6 features. Binary features are shown as within-group
    proportions (bars); continuous features as violins (split for 2 groups; half-violin on the right
    with jittered points on the left for >2 groups). Median lines are drawn on every violin. Each
    panel is annotated with the FDR significance star, plus the effect size only when significant."""
    features = [c for c in data.columns if c not in {"Is Sharing Data", "Sharing Class"}]
    binary = [f for f in features if f in BINARY_FEATURES]
    continuous = [f for f in features if f not in BINARY_FEATURES]
    panel_features = continuous + binary
    subplot_names = list(map(lambda f: _SUBTITLE_MAPPING.get(f, f), panel_features))
    two_groups = len(group_order) == 2

    def group_label(grp):
        if group_feature == "Is Sharing Data":
            return "SHARING" if grp == 1 else "NOT SHARING"
        return str(grp)

    def add_sig_annotation(ftr, fgr, r, c):
        x_pos, y_pos1 = 0.5, 0.925
        sig = results_df.loc[ftr, "sig"]
        assert isinstance(sig, str), f"feature {ftr} has unexpected significance marker {sig} of type {type(sig)}"
        sig_notation = f"<b>{sig}</b>" if sig in ("*", "**", "***") else sig
        fgr.add_annotation(
            row=r, col=c, showarrow=False,
            text=sig_notation, font=AXIS_TICK_FONT,
            xref="x domain", x=x_pos, xanchor="center",
            yref="y domain", y=y_pos1, yanchor="middle",
        )
        if sig not in ("*", "**", "***"):
            return
        # add effect size notation
        es_name, es_val = next(iter(results_df.loc[ftr, "effect_sizes"].items()))
        es_notation = _EFFECT_NOTATION.get(es_name, lambda v: f"{es_name} = {v:.2f}")(es_val)
        fgr.add_annotation(
            row=r, col=c, showarrow=False,
            text=es_notation, font=AXIS_TICK_FONT,
            xref="x domain", x=x_pos, xanchor="center",
            yref="y domain", y=y_pos1, yanchor="top",
        )
        return

    fig = make_subplots(
        rows=2, cols=3, subplot_titles=subplot_names,
        vertical_spacing=0.1, horizontal_spacing=0.05,
    )
    for idx, feature in enumerate(panel_features):
        row, col = (1 if feature in continuous else 2), (idx % 3) + 1
        if feature in _Y_AXIS_MAPPING:
            fig.update_yaxes(
                row=row, col=col,
                title=dict(text=_Y_AXIS_MAPPING[feature], font=AXIS_TITLE_FONT, standoff=4 if col == 1 else 0)
            )
        # set y-axis boundaries to accommodate annotations:
        if feature in BINARY_FEATURES:
            y_low, y_high = 0, 100
        else:
            y_low = min(data[feature]) - 0.15
            y_high = max(data[feature]) + 0.35 if feature == "log(Weeks Since Pub.)" else max(data[feature]) + 1.2
        fig.update_yaxes(row=row, col=col, range=[y_low, y_high])
        add_sig_annotation(feature, fig, row, col)
        # draw the traces:
        for group_idx, group in enumerate(group_order):
            label = group_label(group)
            color = palette[label]
            values = data.loc[data[group_feature] == group, feature]
            show_legend = idx == 0
            if feature in BINARY_FEATURES:
                fig.add_trace(row=row, col=col, trace=go.Bar(
                    x=[label.title()], y=[100 * values.mean()],
                    name=label.title(), legendgroup=label,
                    marker=dict(color=color),
                    showlegend=show_legend,
                ))
            elif two_groups:
                fig.add_trace(row=row, col=col, trace=go.Violin(
                    x=[""] * len(values), y=values,
                    name=label.title(), legendgroup=label,
                    side="positive" if group_idx == 1 else "negative",
                    fillcolor=color, line=dict(color=color),
                    box=dict(visible=False, width=0.9, line=dict(color="black")),
                    meanline=dict(visible=False, color="black"),
                    spanmode="hard", width=0.9, points=False,
                    showlegend=show_legend,
                ))
            else:
                fig.add_trace(row=row, col=col, trace=go.Violin(
                    x=[label.title()] * len(values), y=values,
                    name=label.title(), legendgroup=label,
                    fillcolor=color, line=dict(color=color),
                    box=dict(visible=False, width=0.9, line=dict(color="black")),
                    meanline=dict(visible=False, color="black"),
                    side="positive", points="all", pointpos=-0.25, jitter=0.2,
                    spanmode="hard", width=0.9,
                    showlegend=show_legend,
                ))

    fig.update_annotations(font=AXIS_TITLE_FONT)
    fig.update_xaxes(tickfont=AXIS_TICK_FONT)
    fig.update_xaxes(row=1, showticklabels=False, ticks="")     # hide x-ticks in top-row subplots
    fig.update_yaxes(tickfont=AXIS_TICK_FONT)
    fig.update_yaxes(row=2, showticklabels=False)               # hide y-ticks in bottom-row subplots
    fig.update_yaxes(                                           # show y-ticks and axis title in bottom-left axis
        row=2, col=1, showticklabels=True, ticks="outside",
        title=dict(text="% Articles", font=AXIS_TITLE_FONT, standoff=4)
    )
    fig.update_layout(
        width=1000, height=600,
        title=dict(text=main_title, font=TITLE_FONT, x=0.5, xanchor="center", y=1.0, yanchor="top"),
        legend=dict(
            visible=False, orientation="h", font=AXIS_TICK_FONT,
            x=0.1, xanchor="center", y=1.1, yanchor="top",
        ),
        violinmode="overlay",
        margin=dict(t=50, b=0, l=0, r=0),
        template="simple_white",
    )
    return fig

### Compare Sharing/Non-Sharing

In [ ]:
sharing_comparison_results = dict()
for feature in FEATURES_DF.columns:
    if feature in {"Is Sharing Data", "Sharing Class"}:
        continue
    if feature in BINARY_FEATURES:
        res = compare_binary(data=FEATURES_DF, share_feature="Is Sharing Data", tested_feature=feature, verbose=False)
    else:
        res = compare_continuous(
            data=FEATURES_DF, share_feature="Is Sharing Data", tested_feature=feature, alternative="two-sided", verbose=False
        )
    sharing_comparison_results[feature] = res

# apply corrections
sharing_comparison_results_df = (
    pd.DataFrame.from_dict({k: v[0] for k, v in sharing_comparison_results.items()}, orient="index")
    .rename(columns={"p_val": "p_unc"})
)
sharing_comparison_results_df["p_fdr"] = multipletests(sharing_comparison_results_df["p_unc"], method="fdr_bh")[1]
sharing_comparison_results_df["sig"] = sharing_comparison_results_df["p_fdr"].map(significance_class)

# show results
sharing_comparison_results_df

In [ ]:
sharing_vs_nonsharing_fig = plot_feature_comparison(
    data=FEATURES_DF,
    group_feature="Is Sharing Data",
    group_order=[0, 1],
    results_df=sharing_comparison_results_df,
    main_title="<b>Sharing vs. Non-Sharing across Features</b>",
)

if False:
    sharing_vs_nonsharing_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(sharing_vs_nonsharing_fig, "sharing_vs_nonsharing.png", width=1000, height=600)

### Compare Sharing Granularity

In [ ]:
SHARING_GRANULARITY_SUBSET = FEATURES_DF.loc[FEATURES_DF["Sharing Class"] != "NONE"]
sharing_granularity_results = dict()
for feature in SHARING_GRANULARITY_SUBSET.columns:
    if feature in {"Is Sharing Data", "Sharing Class"}:
        continue
    if feature in BINARY_FEATURES:
        res = compare_binary(
            data=SHARING_GRANULARITY_SUBSET, share_feature="Sharing Class", tested_feature=feature, verbose=False
        )
    else:
        res = compare_continuous(
            data=SHARING_GRANULARITY_SUBSET,
            share_feature="Sharing Class",
            tested_feature=feature,
            alternative="two-sided",
            verbose=False
        )
    sharing_granularity_results[feature] = res

# apply corrections
sharing_granularity_results_df = (
    pd.DataFrame.from_dict({k: v[0] for k, v in sharing_granularity_results.items()}, orient="index")
    .rename(columns={"p_val": "p_unc"})
)
sharing_granularity_results_df["p_fdr"] = multipletests(sharing_granularity_results_df["p_unc"], method="fdr_bh")[1]
sharing_granularity_results_df["sig"] = sharing_granularity_results_df["p_fdr"].map(significance_class)

# show results
sharing_granularity_results_df

In [ ]:
# verify the unique values in the table above:
print("Is `log(Weeks Since Pub.)` normally distributed for ANOVA?")
all_normal = True
for sc in SHARING_GRANULARITY_SUBSET["Sharing Class"].unique():
    subset = SHARING_GRANULARITY_SUBSET.loc[SHARING_GRANULARITY_SUBSET["Sharing Class"] == sc]
    print(f"Sharing Class: {sc}\t(N={len(subset)})")
    shapiro = stats.shapiro(subset["log(Weeks Since Pub.)"])
    all_normal &= shapiro.pvalue > 0.05
    print(f"log publication age is normal:{'Yes' if shapiro.pvalue > 0.05 else 'No'}\t(W={shapiro.statistic :.4f}, p={shapiro.pvalue :.4f})")
    print("####################")
print(f"All distributions are normal: {'Yes' if all_normal else 'No'}")

print("------------")
print("Does `Is Open Access` have low *expected* frequencied for Fisher's Exact?")
contingency_table = pd.crosstab(SHARING_GRANULARITY_SUBSET["Sharing Class"], SHARING_GRANULARITY_SUBSET["Is Open Access"])
res = stats.chi2_contingency(contingency_table, correction=False)
print(f"Expected frequencies below 5: {'Yes' if res[3].min() <= 5 else 'No'}")
res[3]

In [ ]:
# post hoc tests for "Has US Author"
significant_features = sharing_granularity_results_df.loc[sharing_granularity_results_df["p_fdr"]< 0.05].index.tolist()
pairs = list(combinations(SHARING_GRANULARITY_SUBSET["Sharing Class"].unique(), 2))
post_hoc_results = dict()
for feature in significant_features:
    for p in pairs:
        subset = SHARING_GRANULARITY_SUBSET[SHARING_GRANULARITY_SUBSET["Sharing Class"].isin(p)]
        if feature in BINARY_FEATURES:
            res = compare_binary(
                subset, share_feature="Sharing Class", tested_feature=feature, verbose=False
            )[0]
        else:
            res = compare_continuous(
                data=subset, share_feature="Sharing Class", tested_feature=feature,
                alternative="two-sided", verbose=False
            )[0]
        post_hoc_results[(feature, p[0], p[1])] = res

post_hoc_results_df = (
    pd.DataFrame.from_dict(post_hoc_results, orient="index")
    .rename(columns={"p_val": "p_unc"})
)
post_hoc_results_df["p_fdr"] = multipletests(post_hoc_results_df["p_unc"], method="fdr_bh")[1]
post_hoc_results_df["sig"] = post_hoc_results_df["p_fdr"].map(significance_class)
post_hoc_results_df.index.names = ["feature", "group a", "group b"]
post_hoc_results_df = post_hoc_results_df.reset_index(drop=False)
post_hoc_results_df

In [ ]:
granularity_comparison_fig = plot_feature_comparison(
    data=SHARING_GRANULARITY_SUBSET,
    group_feature="Sharing Class",
    group_order=[c for c in SHARING_CLASS_ORDER if c != "NONE"],
    results_df=sharing_granularity_results_df,
    main_title="<b>Sharing Granularity across Features</b>",
)

if False:
    granularity_comparison_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(granularity_comparison_fig, "granularity_comparison.png", width=1000, height=600)